# Notebook 02 — Prétraitement des données
## Challenge ENS #45 — Détection d'apnées du sommeil (Dreem)

---

## Objectif

Ce notebook transforme les données brutes en tableaux NumPy normalisés et séparés train/val/test,
prêts pour l'entraînement PyTorch du notebook 03.

**Pipeline :**
```
X_train.h5  →  split par sujet  →  StandardScaler  →  X_train_processed.npy
                                                    →  X_val_processed.npy
X_test.h5   →  StandardScaler  →  X_test_processed.npy
y_train.csv →  split par sujet  →  y_train_processed.npy
                                →  y_val_processed.npy
```

| Étape | Section | Justification |
|-------|---------|---------------|
| Chargement | §2 | h5py + pandas |
| Split par sujet | §3 | Évite la fuite de données inter-sujets |
| Analyse avant normalisation | §4 | Visualise les amplitudes brutes |
| StandardScaler | §5 | Uniformise les échelles des 8 canaux |
| Calcul pos_weight | §6 | Informe BCEWithLogitsLoss du notebook 03 |
| Sauvegarde + vérification | §7-8 | Cohérence et absence de NaN |


---

## 1. Imports & configuration


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import h5py
import pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({'figure.dpi': 100})

print(f'numpy   : {np.__version__}')
import sklearn; print(f'sklearn : {sklearn.__version__}')


In [ ]:
# ── Chemins ──────────────────────────────────────────────────────────────────
RAW_DIR     = Path('../data/raw/')
PROC_DIR    = Path('../data/processed/')
FIGURES_DIR = Path('../figures/')
PROC_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

X_TRAIN_H5  = RAW_DIR / 'X_train.h5'
X_TEST_H5   = RAW_DIR / 'X_test.h5'
Y_TRAIN_CSV = RAW_DIR / 'y_train_tX9Br0C.csv'

print(f'RAW_DIR  : {RAW_DIR.resolve()}')
print(f'PROC_DIR : {PROC_DIR.resolve()}')


In [ ]:
# ── Constantes (cohérentes avec config.py et notebook 03) ────────────────────
RANDOM_SEED       = 42
N_SIGNALS         = 8
N_SAMPLES         = 9000
N_LABEL_COLS      = 90
SIGNAL_FREQ       = 100
WINDOW_SECS       = 90
N_WINDOWS_PER_SUB = 200

SIGNAL_NAMES = [
    'AbdoBelt', 'AirFlow', 'PPG', 'ThorBelt',
    'Snoring', 'SpO2', 'EEG C4-A1', 'EEG O2-A1'
]

# Sujets de validation — choix guidé par l'EDA (notebook 01)
# Taux d'apnée : sujet 0 = 0.99%, 1 = 2.90%, 8 = 13.77%, 13 = 1.63%, 15 = 3.41%
# → Couvre une gamme de sévérité, représentatif sans être biaisé vers les cas sévères
VAL_SUBJECTS = [0, 1, 8, 13, 15]

np.random.seed(RANDOM_SEED)
print(f'Sujets de validation : {VAL_SUBJECTS}')


---

## 2. Chargement des données brutes


In [ ]:
# ── Chargement X_train ───────────────────────────────────────────────────────
print('Chargement X_train.h5 ...')
with h5py.File(X_TRAIN_H5, 'r') as f:
    key = list(f.keys())[0]
    print(f'  Clé H5 : "{key}"  shape : {f[key].shape}  dtype : {f[key].dtype}')
    X_all = f[key][:].astype(np.float32)

print('Chargement X_test.h5 ...')
with h5py.File(X_TEST_H5, 'r') as f:
    key = list(f.keys())[0]
    print(f'  Clé H5 : "{key}"  shape : {f[key].shape}  dtype : {f[key].dtype}')
    X_test_raw = f[key][:].astype(np.float32)

print('Chargement y_train ...')
df_labels  = pd.read_csv(Y_TRAIN_CSV)
label_cols = [f'y_{i}' for i in range(N_LABEL_COLS)]
Y_all      = df_labels[label_cols].values.astype(np.float32)

N_SUBJECTS  = X_all.shape[0] // N_WINDOWS_PER_SUB
subject_ids = np.arange(X_all.shape[0]) // N_WINDOWS_PER_SUB

print(f'\nRésumé :')
print(f'  X_all      : {X_all.shape}  ({X_all.nbytes/1e6:.0f} Mo)')
print(f'  X_test_raw : {X_test_raw.shape}  ({X_test_raw.nbytes/1e6:.0f} Mo)')
print(f'  Y_all      : {Y_all.shape}')
print(f'  Sujets     : {N_SUBJECTS}')


---

## 3. Split train / validation par sujet

### Pourquoi splitter par sujet et non aléatoirement ?

Un **split aléatoire au niveau des fenêtres** créerait une fuite de données :
- Les fenêtres d'un même sujet sont hautement corrélées (même morphologie des signaux,
  même sévérité d'apnée, mêmes artefacts)
- Un modèle entraîné sur 80% des fenêtres d'un sujet "mémoriserait" ses patterns
  et obtiendrait un F1 artificiellement élevé sur les 20% restants
- Cela ne simule pas la situation réelle : les 22 sujets de test sont **entièrement inconnus**

Un **split par sujet** (`GroupKFold`) simule correctement ce scénario.

**Split retenu :**
- **Train :** 17 sujets → 3400 fenêtres (~78%)
- **Validation :** 5 sujets → 1000 fenêtres (~22%)


In [ ]:
# ── Construction des masques train/val ───────────────────────────────────────
val_mask   = np.isin(subject_ids, VAL_SUBJECTS)
train_mask = ~val_mask
train_idx  = np.where(train_mask)[0]
val_idx    = np.where(val_mask)[0]

train_subjects = sorted(set(subject_ids[train_idx].tolist()))
val_subjects   = sorted(set(subject_ids[val_idx].tolist()))

print(f'Sujets TRAIN ({len(train_subjects)}) : {train_subjects}')
print(f'Sujets VAL   ({len(val_subjects)})  : {val_subjects}')
print(f'\nFenêtres TRAIN : {len(train_idx)}')
print(f'Fenêtres VAL   : {len(val_idx)}')

# Vérification : aucun sujet en commun
assert set(train_subjects) & set(val_subjects) == set(), 'FUITE TRAIN→VAL !'
print('\nAssert pas de fuite : OK')


In [ ]:
# ── Extraction des sous-ensembles ────────────────────────────────────────────
X_train_split = X_all[train_idx]   # (3400, 8, 9000)
X_val_split   = X_all[val_idx]     # (1000, 8, 9000)
y_train_split = Y_all[train_idx]   # (3400, 90)
y_val_split   = Y_all[val_idx]     # (1000, 90)

print(f'X_train_split : {X_train_split.shape}')
print(f'X_val_split   : {X_val_split.shape}')
print(f'y_train_split : {y_train_split.shape}')
print(f'y_val_split   : {y_val_split.shape}')


In [ ]:
# ── Visualisation du split ───────────────────────────────────────────────────
apnea_rates = np.array([
    Y_all[subject_ids == s].mean() * 100 for s in range(N_SUBJECTS)
])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Répartition du split train/validation', fontsize=11)

# Taux d'apnée par sujet + split
colors = ['#FF9800' if s in VAL_SUBJECTS else '#2196F3' for s in range(N_SUBJECTS)]
axes[0].bar(range(N_SUBJECTS), apnea_rates, color=colors, edgecolor='white')
axes[0].axhline(apnea_rates[[s for s in range(N_SUBJECTS) if s not in VAL_SUBJECTS]].mean(),
                color='#2196F3', linestyle='--', label=f'Moy train = {apnea_rates[[s for s in range(N_SUBJECTS) if s not in VAL_SUBJECTS]].mean():.2f}%')
axes[0].axhline(apnea_rates[VAL_SUBJECTS].mean(),
                color='#FF9800', linestyle='--', label=f'Moy val = {apnea_rates[VAL_SUBJECTS].mean():.2f}%')
axes[0].set_xlabel('Sujet')
axes[0].set_ylabel("Taux d'apnée (%)")
axes[0].set_title('Taux apnée par sujet')
tp = mpatches.Patch(color='#2196F3', label='Train')
vp = mpatches.Patch(color='#FF9800', label='Val')
axes[0].legend(handles=[tp, vp])
axes[0].grid(alpha=0.3)

# Résumé chiffré
train_r = apnea_rates[[s for s in range(N_SUBJECTS) if s not in VAL_SUBJECTS]]
val_r   = apnea_rates[VAL_SUBJECTS]
axes[1].bar(['Train\n(17 sujets)', 'Val\n(5 sujets)'],
            [train_r.mean(), val_r.mean()],
            color=['#2196F3', '#FF9800'], edgecolor='white', width=0.4)
axes[1].errorbar(['Train\n(17 sujets)', 'Val\n(5 sujets)'],
                 [train_r.mean(), val_r.mean()],
                 yerr=[train_r.std(), val_r.std()],
                 fmt='none', color='black', capsize=5)
axes[1].set_ylabel("Taux d'apnée moyen (%) ± std")
axes[1].set_title('Biais du split (val plus facile)')
axes[1].grid(alpha=0.3, axis='y')
axes[1].text(0.5, max(train_r.mean(), val_r.mean()) * 0.7,
             'Note : val biaisée vers sujets légers\n→ F1_val sera légèrement optimiste',
             ha='center', fontsize=9, style='italic',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_10_split_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


---

## 4. Analyse des amplitudes brutes (avant normalisation)

On visualise les distributions avant normalisation pour justifier le choix de StandardScaler.


In [ ]:
# ── Distributions brutes par canal ───────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('Amplitudes brutes par canal — X_train (avant normalisation)', fontsize=11)

COLORS = ['#2196F3','#F44336','#4CAF50','#9C27B0','#FF9800','#00BCD4','#795548','#607D8B']

rng = np.random.default_rng(RANDOM_SEED)
for ch, (ax, name, color) in enumerate(zip(axes.ravel(), SIGNAL_NAMES, COLORS)):
    data = X_train_split[:, ch, :].ravel()
    sample = rng.choice(data, size=min(30000, len(data)), replace=False)
    ax.hist(sample, bins=70, color=color, alpha=0.75, edgecolor='none')
    ax.set_title(f'{name}\nμ={sample.mean():.2f}  σ={sample.std():.2f}', fontsize=9)
    ax.set_ylabel('Fréquence')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_11_raw_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Observations :')
print('  → Amplitudes très différentes entre canaux → normalisation obligatoire')
print('  → Certains canaux ont des queues lourdes → StandardScaler plus robuste que MinMaxScaler')


---

## 5. Normalisation — StandardScaler par canal

### Pourquoi StandardScaler ?

| Méthode | Principe | Avantage | Inconvénient |
|---------|----------|----------|---------------|
| **StandardScaler** | x → (x - μ) / σ | Robuste aux queues, pas de bornes | Sensible aux outliers extrêmes |
| MinMaxScaler | x → (x - min) / (max - min) | Bornes [0,1] garanties | Très sensible aux outliers |
| RobustScaler | x → (x - médiane) / IQR | Très robuste aux outliers | Plus complexe |

**Choix : StandardScaler** — les signaux PSG ont des distributions approximativement
gaussiennes avec des queues modérées, StandardScaler est adapté et simple.

### Règle absolue : fit sur TRAIN uniquement

Le scaler est ajusté **exclusivement sur X_train_split**. Utiliser val ou test pour
calculer μ et σ constituerait une fuite de données (le modèle verrait indirectement
la distribution du test avant l'inférence).


In [ ]:
# ── Fit du scaler ─────────────────────────────────────────────────────────────
N_train = X_train_split.shape[0]
N_val   = X_val_split.shape[0]
N_test  = X_test_raw.shape[0]

# Reshape (N, 8, 9000) → (N×9000, 8) pour sklearn
X_tr_2d = X_train_split.transpose(0, 2, 1).reshape(-1, N_SIGNALS)  # (3400×9000, 8)
X_va_2d = X_val_split.transpose(0, 2, 1).reshape(-1, N_SIGNALS)
X_te_2d = X_test_raw.transpose(0, 2, 1).reshape(-1, N_SIGNALS)

print('Fitting StandardScaler sur X_train_split uniquement ...')
scaler = StandardScaler()
scaler.fit(X_tr_2d)

print(f'\n{"Canal":<15} {"μ_train":>10} {"σ_train":>10}')
print('-' * 38)
for name, mu, sig in zip(SIGNAL_NAMES, scaler.mean_, scaler.scale_):
    print(f'{name:<15} {mu:>10.4f} {sig:>10.4f}')


In [ ]:
# ── Application du scaler ─────────────────────────────────────────────────────
print('Transformation ...')

X_tr_norm = scaler.transform(X_tr_2d).reshape(N_train, N_SAMPLES, N_SIGNALS)
X_tr_norm = X_tr_norm.transpose(0, 2, 1).astype(np.float32)   # (3400, 8, 9000)

X_va_norm = scaler.transform(X_va_2d).reshape(N_val, N_SAMPLES, N_SIGNALS)
X_va_norm = X_va_norm.transpose(0, 2, 1).astype(np.float32)   # (1000, 8, 9000)

X_te_norm = scaler.transform(X_te_2d).reshape(N_test, N_SAMPLES, N_SIGNALS)
X_te_norm = X_te_norm.transpose(0, 2, 1).astype(np.float32)   # (4400, 8, 9000)

# Vérification : μ ≈ 0, σ ≈ 1 sur train
check_2d = X_tr_norm.transpose(0, 2, 1).reshape(-1, N_SIGNALS)
print(f'\nVérification sur X_train normalisé :')
print(f'  Moyennes : {check_2d.mean(axis=0).round(6)}')
print(f'  Stds     : {check_2d.std(axis=0).round(4)}')
print('  (doivent être ≈ 0.0 et ≈ 1.0)')


In [ ]:
# ── Avant / Après normalisation ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('Distributions après normalisation — X_train', fontsize=11)

rng2 = np.random.default_rng(RANDOM_SEED)
for ch, (ax, name, color) in enumerate(zip(axes.ravel(), SIGNAL_NAMES, COLORS)):
    data = X_tr_norm[:, ch, :].ravel()
    sample = rng2.choice(data, size=min(30000, len(data)), replace=False)
    ax.hist(sample, bins=70, color=color, alpha=0.75, edgecolor='none')
    ax.axvline(0, color='black', linewidth=1.5, linestyle='--')
    ax.set_title(f'{name}\nμ={sample.mean():.4f}  σ={sample.std():.4f}', fontsize=9)
    ax.set_ylabel('Fréquence')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_12_normalized_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('→ Toutes les distributions sont centrées en 0 avec σ ≈ 1')


---

## 6. Calcul du pos_weight sur le split train

Le `pos_weight` est calculé **uniquement sur les 17 sujets du split train**
(3400 fenêtres, 306 000 secondes). Cette valeur est ensuite utilisée dans
`BCEWithLogitsLoss(pos_weight=...)` du notebook 03.

$$w^+ = \frac{N_{\text{négatifs}}}{N_{\text{positifs}}} = \frac{282\,872}{23\,128} \approx 12.23$$

> **Note :** La valeur EDA globale (22 sujets) était ≈ 13.6. On utilise la valeur
> **split-train** pour cohérence : le modèle est entraîné uniquement sur ces 17 sujets.


In [ ]:
# ── Calcul du pos_weight ─────────────────────────────────────────────────────
n_pos      = int(y_train_split.sum())
n_neg      = y_train_split.size - n_pos
pos_weight = n_neg / n_pos
rate_pct   = n_pos / y_train_split.size * 100

print('='*55)
print('  DÉSÉQUILIBRE — split TRAIN (17 sujets, 3400 fenêtres)')
print('='*55)
print(f'  N+ (secondes apnée)    : {n_pos:>10,}  ({rate_pct:.2f}%)')
print(f'  N- (secondes normales) : {n_neg:>10,}  ({100-rate_pct:.2f}%)')
print(f'  pos_weight             : {pos_weight:.4f}  ≈ {pos_weight:.2f}')
print(f'  → POS_WEIGHT = {pos_weight:.2f} dans config.py et notebook 03')

# Fenêtres positives
pct_pos_win = (y_train_split.sum(axis=1) > 0).mean() * 100
pct_pos_win_val = (y_val_split.sum(axis=1) > 0).mean() * 100
print(f'\n  Fenêtres positives TRAIN : {pct_pos_win:.1f}%')
print(f'  Fenêtres positives VAL   : {pct_pos_win_val:.1f}%')


In [ ]:
# ── Visualisation du déséquilibre par sujet (train seulement) ─────────────────
train_subs_list = [s for s in range(N_SUBJECTS) if s not in VAL_SUBJECTS]
train_rates_by_sub = [
    Y_all[subject_ids == s].mean() * 100 for s in train_subs_list
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(train_subs_list)), train_rates_by_sub,
            color='#2196F3', edgecolor='white')
axes[0].axhline(np.mean(train_rates_by_sub), color='firebrick', linestyle='--',
                label=f'Moy = {np.mean(train_rates_by_sub):.2f}%')
axes[0].set_xticks(range(len(train_subs_list)))
axes[0].set_xticklabels([f'S{s}' for s in train_subs_list], rotation=45)
axes[0].set_xlabel('Sujet')
axes[0].set_ylabel("Taux d'apnée (%)")
axes[0].set_title('Taux apnée par sujet — TRAIN uniquement')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Camembert pos_weight
axes[1].pie([n_pos, n_neg],
            labels=[f'Apnée\n{n_pos:,} s\n({rate_pct:.1f}%)',
                    f'Normal\n{n_neg:,} s\n({100-rate_pct:.1f}%)'],
            colors=['#F44336', '#2196F3'], autopct='%1.1f%%',
            explode=(0.05, 0), startangle=90)
axes[1].set_title(f'Déséquilibre train\npos_weight = {pos_weight:.2f}')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_13_train_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()


---

## 7. Sauvegarde des fichiers traités

Les tableaux sont sauvegardés en `.npy` pour un accès rapide via
`np.load(..., mmap_mode='r')` dans le notebook 03 (lazy loading, 2.5 Go total).


In [ ]:
# ── Sauvegarde .npy ──────────────────────────────────────────────────────────
files_to_save = {
    'X_train_processed.npy' : X_tr_norm,
    'X_val_processed.npy'   : X_va_norm,
    'X_test_processed.npy'  : X_te_norm,
    'y_train_processed.npy' : y_train_split,
    'y_val_processed.npy'   : y_val_split,
    'train_idx.npy'         : train_idx,
    'val_idx.npy'           : val_idx,
}

print('Sauvegarde ...')
for fname, arr in files_to_save.items():
    path = PROC_DIR / fname
    np.save(path, arr)
    print(f'  {fname:35s} {str(arr.shape):20s} {path.stat().st_size/1e6:6.1f} Mo')

# Scaler
scaler_path = PROC_DIR / 'scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'  {"scaler.pkl":35s} {"":20s} {scaler_path.stat().st_size/1e3:6.1f} Ko')

total_gb = sum((PROC_DIR/f).stat().st_size for f in
    ['X_train_processed.npy','X_val_processed.npy','X_test_processed.npy']) / 1e9
print(f'\nTaille totale .npy (signaux) : {total_gb:.2f} Go')


---

## 8. Vérifications de cohérence


In [ ]:
# ── Vérification 1 : rechargement ─────────────────────────────────────────────
print('Test rechargement depuis disque ...')
X_tr_check = np.load(PROC_DIR / 'X_train_processed.npy', mmap_mode='r')
y_tr_check = np.load(PROC_DIR / 'y_train_processed.npy')

assert X_tr_check.shape == (3400, 8, 9000), f'Shape inattendu : {X_tr_check.shape}'
assert y_tr_check.shape == (3400, 90),      f'Shape inattendu : {y_tr_check.shape}'
assert np.allclose(X_tr_check[0], X_tr_norm[0], atol=1e-5), 'Incohérence !'
print('  Shapes   : OK')
print('  Valeurs  : OK (allclose, atol=1e-5)')

# ── Vérification 2 : NaN / Inf ────────────────────────────────────────────────
print('\nTest NaN/Inf ...')
for name, arr in [('X_train', X_tr_norm), ('X_val', X_va_norm), ('X_test', X_te_norm)]:
    n_nan = np.isnan(arr).sum()
    n_inf = np.isinf(arr).sum()
    status = 'OK' if n_nan == 0 and n_inf == 0 else 'PROBLEME'
    print(f'  {name:10s} — NaN: {n_nan}  Inf: {n_inf}  → {status}')

# ── Vérification 3 : pas de fuite sujet ──────────────────────────────────────
print('\nTest fuite sujet ...')
assert set(train_subjects) & set(val_subjects) == set()
print(f'  Sujets train : {train_subjects}')
print(f'  Sujets val   : {val_subjects}')
print('  Pas de fuite : OK')

# ── Vérification 4 : normalisation ───────────────────────────────────────────
print('\nTest normalisation ...')
check_2d = X_tr_norm.transpose(0,2,1).reshape(-1, N_SIGNALS)
means = check_2d.mean(axis=0)
stds  = check_2d.std(axis=0)
print(f'  Moyennes train : {means.round(6)}  (doivent ≈ 0)')
print(f'  Stds train     : {stds.round(4)}   (doivent ≈ 1)')
assert np.allclose(means, 0, atol=1e-4) and np.allclose(stds, 1, atol=1e-3)
print('  OK')


In [ ]:
# ── Résumé final ──────────────────────────────────────────────────────────────
print('=' * 60)
print('  RÉSUMÉ DU PRÉTRAITEMENT')
print('=' * 60)
print(f'  X_train : (3400, 8, 9000) — 17 sujets')
print(f'  X_val   : (1000, 8, 9000) — 5 sujets {VAL_SUBJECTS}')
print(f'  X_test  : (4400, 8, 9000) — 22 sujets inconnus')
print(f'  y_train : (3400, 90)  float32')
print(f'  y_val   : (1000, 90)  float32')
print(f'  pos_weight (train) = {pos_weight:.2f}')
print(f'  Scaler : StandardScaler (fitté sur train)')
print()
print('  Tous les fichiers sont dans :', PROC_DIR.resolve())
print()
print('  → Le notebook 03 charge ces fichiers avec mmap_mode="r" (lazy loading)')
print('  → POS_WEIGHT = 12.23 à reporter dans config.py et notebook 03')


---

## 9. Note sur le biais de validation et perspectives

### Biais du split actuel

Les 5 sujets de validation ont un taux d'apnée moyen de **~4.5%**, nettement
inférieur à la moyenne des sujets d'entraînement (~7.6%).
Le F1 de validation mesuré dans le notebook 03 sera donc **légèrement optimiste**.

### Solution : GroupKFold

```python
from sklearn.model_selection import GroupKFold

groups = np.arange(X_all.shape[0]) // N_WINDOWS_PER_SUB   # sujet de chaque fenêtre
gkf    = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_all, Y_all, groups)):
    # 5 splits différents, chaque sujet apparaît 1× en val
    # F1 moyen sur 5 folds = estimation robuste des performances
    ...
```

Cette approche multiplie le temps d'entraînement par 5 mais donne une estimation
fiable de la performance sur des sujets inconnus — ce qui est l'objectif du challenge.
Elle est référencée dans `config.py` (`N_FOLDS = 5`).
